# 07 · Scout exploration and diversity energy

**Status: UNEXECUTED FIRST DRAFT.** This notebook contains working implementations to inspect and refine when run. No trained capability or benchmark result is asserted.

**Purpose:** Explore alternate action hypotheses using a second conditional energy model, larger latent proposals and explicit diversity pressure.

**Inputs:** Notebook 06 proposal/energy definitions and public legal action mask.

**Outputs:** ScoutEBM with learned proposal heads, energy refinement, diversity and uncertainty-ready candidate metadata.

**Expected runtime:** Under a minute for meta checks; measured full-model inference in 09. These are planning estimates; measure on this instance.

**Checkpoint directory:** `checkpoints/<run_id>/` under the project root. Definitions in this notebook are included in every source manifest.

**Exact resume:** Restart the AHR v2 XL kernel and Run All. This notebook defines components and runs bounded checks; it does not start a long training job. Training continuation is handled in 10 and 11 from the latest valid checkpoint.

The configuration remains the full XL design (`d_model=1024` by default). Any explicitly identified synthetic fixture verifies plumbing only and is not a smaller AHR training configuration.


## Load explicit notebook dependencies
Only tagged definitions are loaded; dependency execution cells are not run.

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([6], globals())

## Scout model

Scout keeps its own parameters and starts farther from the current semantic context. Pairwise repulsion encourages different hypotheses without declaring them correct. World-model uncertainty is combined with Scout candidates by the planner; it is not evidence of novelty by itself.

In [ ]:
class ScoutEBM(WorkerEBM):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.diversity_weight = 0.1
        self.novelty_weight = 0.05
        self.refine_rate = 0.12

    def propose(self, context, goal, mask, count):
        validate_action_mask(mask, context.shape[0])
        with torch.no_grad():
            candidates = self.proposal_heads.candidates(context, goal, count, noise_scale=0.35)
        for _ in range(self.refine_steps):
            with torch.enable_grad(), torch.autocast(device_type=context.device.type, enabled=False):
                variable = candidates.detach().float().requires_grad_(True)
                energies = self.energy(context.detach().float(), variable, goal.detach().float())
                novelty = (variable-context.detach().float().unsqueeze(1)).square().mean(dim=-1)
                distances = torch.cdist(variable, variable).square()/variable.shape[-1]
                count_now = variable.shape[1]
                if count_now > 1:
                    off_diagonal = ~torch.eye(count_now, device=variable.device, dtype=torch.bool).unsqueeze(0)
                    repulsion = torch.exp(-distances).masked_select(off_diagonal.expand_as(distances)).mean()
                else:
                    repulsion = variable.sum()*0.0
                objective = energies.mean() - self.novelty_weight*novelty.mean() + self.diversity_weight*repulsion
                gradient = torch.autograd.grad(objective, variable, create_graph=False)[0]
                gradient = gradient / gradient.norm(dim=-1, keepdim=True).clamp_min(1.0)
                candidates = (variable-self.refine_rate*gradient).clamp(-self.latent_clip, self.latent_clip).detach()
        with torch.no_grad():
            candidates = candidates.to(context.dtype)
            actions, xy, logits, click_logits = self.proposal_heads.decode(candidates, mask)
            energies = self.energy(context, candidates, goal)
            novelty = (candidates-context.unsqueeze(1)).float().square().mean(dim=-1)
        return {'latents': candidates, 'actions': actions, 'xy': xy, 'energies': energies,
                'novelty': novelty, 'action_logits': logits, 'click_logits': click_logits}

## Run All: bounded validation

The parameter check confirms Scout is a full independent energy/proposal network. This does not train it or show useful exploration.

In [ ]:
cfg = load_config()
with torch.device('meta'):
    scout_metadata = ScoutEBM(cfg)
print({'scout_parameters': sum(p.numel() for p in scout_metadata.parameters()),
       'scout_enabled': cfg['reasoning']['scout_enabled'], 'shared_parameters_with_worker': False})
del scout_metadata
assert 1 <= cfg['reasoning']['scout_candidates'] <= 128